In [ ]:
!pip uninstall torchao -y

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [ ]:
!pip install peft -q
from peft import get_peft_model, LoraConfig, TaskType

In [ ]:
import pandas as pd
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback, AutoModelForSequenceClassification, AutoTokenizer, set_seed, TrainerCallback
from datasets import Dataset
import joblib
import numpy as np
from google.colab import drive
import os
import json
import zipfile
from sklearn.metrics import f1_score, classification_report
import time
import torch
from transformers.trainer_utils import get_last_checkpoint
from transformers import DataCollatorWithPadding


In [ ]:
drive.mount('/content/drive')

SPLIT_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/SCOTBESS_FULL_ANNOTATED_TERRA_LOW_SPLIT.csv"
MLB_PATH = "/content/drive/MyDrive/thesis_results/SCOTBESS_splits/scotbess_mlb.joblib"

output_dir = "/content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_LoRA"
os.makedirs(output_dir, exist_ok=True)


Mounted at /content/drive


Loading the dataset

In [ ]:
scotbess_df = pd.read_csv(SPLIT_PATH)

In [ ]:
scotbess_df_train = scotbess_df[scotbess_df["split"] == "train"].reset_index(drop=True)
scotbess_df_val = scotbess_df[scotbess_df["split"] == "validation"].reset_index(drop=True)
scotbess_df_test = scotbess_df[scotbess_df["split"] == "test"].reset_index(drop=True)

print("Train shape:", scotbess_df_train.shape)
print("Validation shape:", scotbess_df_val.shape)
print("Test shape:", scotbess_df_test.shape)

Train shape: (1340, 7)
Validation shape: (165, 7)
Test shape: (170, 7)


In [ ]:
mlb = joblib.load(MLB_PATH)

print("Number of labels:", len(mlb.classes_))
print(mlb.classes_)

Number of labels: 20
['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']


In [ ]:
def parse_labels(value):
    if isinstance(value, list):
        return value
    if pd.isna(value):
        return []
    return json.loads(value)

for df in [scotbess_df_train, scotbess_df_val, scotbess_df_test]:
    df["label_list"] = df["labels"].apply(parse_labels)


In [ ]:
scotbess_y_train = mlb.transform(scotbess_df_train["label_list"])
scotbess_y_val = mlb.transform(scotbess_df_val["label_list"])
scotbess_y_test = mlb.transform(scotbess_df_test["label_list"])


In [ ]:
scotbess_y_train.shape, scotbess_y_val.shape, scotbess_y_test.shape


((1340, 20), (165, 20), (170, 20))

In [ ]:
scotbess_X_train = scotbess_df_train["final_masked_text"].fillna("").astype(str)
scotbess_X_val = scotbess_df_val["final_masked_text"].fillna("").astype(str)
scotbess_X_test = scotbess_df_test["final_masked_text"].fillna("").astype(str)

In [ ]:
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

In [ ]:
# ModernBERT's token length increased to 8192 (the model's limit) for SCOTBESS
def tokenize(texts):
    return tokenizer(texts.tolist(), truncation=True, max_length=8192)
#dynamic padding
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)

train_enc = tokenize(scotbess_X_train)
dev_enc = tokenize(scotbess_X_val)
test_enc = tokenize(scotbess_X_test)

In [ ]:
y_train_bin = scotbess_y_train.astype(np.float32)
y_dev_bin = scotbess_y_val.astype(np.float32)
y_test_bin = scotbess_y_test.astype(np.float32)

print(mlb.classes_)
print(y_train_bin.shape)


['Agricultural Land' 'Community and Economic Benefits'
 'Consultation, Transparency and Information' 'Cumulative Impact'
 'Decommissioning and Site Restoration' 'Emergency Planning and Response'
 'Fire and Explosion Risk' 'Grid Connection and Electrical Infrastructure'
 'Health and Wellbeing' 'Landscape, Visual and Heritage Impact'
 'Light Pollution' 'Noise' 'Planning Policy and Regulatory Compliance'
 'Project Need' 'Property Value'
 'Residential Proximity and Separation Distance' 'Site Selection'
 'Traffic' 'Water and Soil Contamination' 'Wildlife and Ecology']
(1340, 20)


In [ ]:
id2label = {i: label for i, label in enumerate(mlb.classes_)}
label2id = {label: i for i, label in enumerate(mlb.classes_)}

In [ ]:
train_dataset = Dataset.from_dict({
    "input_ids": train_enc["input_ids"],
    "attention_mask": train_enc["attention_mask"],
    "labels": y_train_bin.astype("float32")})

eval_dataset = Dataset.from_dict({
    "input_ids": dev_enc["input_ids"],
    "attention_mask": dev_enc["attention_mask"],
    "labels": y_dev_bin.astype("float32")})

test_dataset = Dataset.from_dict({
    "input_ids": test_enc["input_ids"],
    "attention_mask": test_enc["attention_mask"],
    "labels": y_test_bin.astype("float32")})

In [ ]:
print(train_dataset[0])
print(len(train_dataset[0]["labels"]))

{'input_ids': [50281, 1231, 452, 4092, 33196, 326, 627, 556, 417, 644, 247, 18731, 22887, 23868, 20023, 1754, 327, 253, 1511, 273, 9378, 281, 320, 908, 275, 253, 4081, 2341, 5718, 9509, 15, 4325, 253, 1491, 12164, 253, 2670, 588, 452, 260, 1884, 35669, 273, 2341, 5718, 407, 247, 2962, 273, 23178, 9378, 5085, 273, 260, 15, 22, 35669, 5350, 15, 3954, 1568, 275, 253, 7177, 1057, 352, 3748, 253, 1511, 273, 9378, 281, 320, 908, 2299, 342, 253, 1655, 4302, 253, 760, 16571, 4500, 651, 320, 27747, 14, 279, 1754, 327, 2341, 4038, 15, 380, 12794, 2495, 273, 2442, 19333, 275, 31976, 1514, 19978, 310, 973, 1929, 285, 973, 14290, 21349, 15, 496, 253, 1982, 835, 9002, 16638, 32560, 27173, 1052, 5718, 9189, 627, 574, 2168, 644, 374, 14, 20, 19333, 15, 844, 2868, 247, 2120, 3907, 2495, 6803, 943, 320, 26237, 1754, 327, 9378, 1511, 313, 284, 359, 476, 760, 5467, 31976, 1514, 428, 42, 251, 10, 347, 247, 9509, 273, 436, 2341, 4038, 1735, 281, 16252, 285, 9787, 3607, 24543, 247, 3289, 2495, 342, 6774, 372

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred

    probs = 1 / (1 + np.exp(-logits))   # sigmoid
    preds = (probs >= 0.5).astype(int)

    labels = labels.astype(int)

    f1_micro = f1_score(labels, preds, average="micro", zero_division=0)
    f1_macro = f1_score(labels, preds, average="macro", zero_division=0)

    return {
        "f1_micro": f1_micro,
        "f1_macro": f1_macro}

In [ ]:
print("Train labels:", y_train_bin.shape)
print("Val labels:", y_dev_bin.shape)
print("Test labels:", y_test_bin.shape)
print("Number of labels:", len(mlb.classes_))

Train labels: (1340, 20)
Val labels: (165, 20)
Test labels: (170, 20)
Number of labels: 20


**Training function**

In [ ]:
#helpers for counting times for the final run with ModernBERT, as the run would get disconnected by colab  since it takes a long time to run it with 10 epochs on AAPD; reusing for SCOTBESS
class CheckpointTimeCallback(TrainerCallback):
    def __init__(self, output_dir):
        self.output_dir = output_dir
        self.time_log_path = os.path.join(output_dir, "time_log.json")
        self.previous_time_sec = 0.0
        self.session_start = None
        self.current_total_time_sec = 0.0
        self.current_session_time_sec = 0.0

    def on_train_begin(self, args, state, control, **kwargs):
        if os.path.exists(self.time_log_path):
            with open(self.time_log_path, "r") as f:
                self.previous_time_sec = json.load(f).get("train_time_sec", 0.0)
        else:
            self.previous_time_sec = 0.0

        self.session_start = time.perf_counter()
        self.current_total_time_sec = self.previous_time_sec
        self.current_session_time_sec = 0.0

    def _save_time(self, state):
        if torch.cuda.is_available():
            torch.cuda.synchronize()

        self.current_session_time_sec = time.perf_counter() - self.session_start
        self.current_total_time_sec = self.previous_time_sec + self.current_session_time_sec

        data = {
            "train_time_sec": self.current_total_time_sec,
            "current_session_train_time_sec": self.current_session_time_sec,
            "previous_train_time_sec": self.previous_time_sec,
            "last_global_step": int(state.global_step),
            "last_epoch": float(state.epoch) if state.epoch is not None else None,
        }

        os.makedirs(self.output_dir, exist_ok=True)

        tmp_path = self.time_log_path + ".tmp"
        with open(tmp_path, "w") as f:
            json.dump(data, f, indent=2)

        os.replace(tmp_path, self.time_log_path)

    def on_save(self, args, state, control, **kwargs):
        self._save_time(state)

    def on_train_end(self, args, state, control, **kwargs):
        self._save_time(state)

    def get_times(self):
        if os.path.exists(self.time_log_path):
            with open(self.time_log_path, "r") as f:
                data = json.load(f)

            return (
                data.get("train_time_sec", self.current_total_time_sec),
                data.get("current_session_train_time_sec", self.current_session_time_sec),
            )

        return self.current_total_time_sec, self.current_session_time_sec

In [ ]:
def sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()

def reset_cuda_peak_memory():
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()

#measure vram only for final best config
def get_peak_vram_gb():
    if not torch.cuda.is_available():
        return None

    torch.cuda.synchronize()
    return torch.cuda.max_memory_allocated() / (1024 ** 3)


def run_training(config, seed=0, evaluate_test=False, measure_vram=False, save_report=False):
    set_seed(seed)


    if measure_vram:
        reset_cuda_peak_memory()


    model = AutoModelForSequenceClassification.from_pretrained(
        config["base_model"],
        num_labels=len(mlb.classes_),
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id,
        attn_implementation="sdpa")

    peft_config = LoraConfig(
        task_type=TaskType.SEQ_CLS,
        r=config["lora_r"],
        lora_alpha=config["lora_alpha"],
        lora_dropout=config["lora_dropout"],
        target_modules=config["target_modules"],
        modules_to_save= config["modules_to_save"])
    model = get_peft_model(model, peft_config)


    #for the final statistics
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total_params = sum(p.numel() for p in model.parameters())

    training_args = TrainingArguments(
        output_dir=config["output_dir"],

        learning_rate=config["learning_rate"],
        per_device_train_batch_size=config["micro_batch_size"],
        per_device_eval_batch_size=config["micro_batch_size"],
        gradient_accumulation_steps=config["gradient_accumulation_steps"],

        num_train_epochs=config["num_train_epochs"],
        weight_decay=config["weight_decay"],
        warmup_ratio=config["warmup_ratio"],

        eval_strategy="epoch",
        save_strategy="epoch",
        logging_strategy="epoch",

        load_best_model_at_end=True,
        metric_for_best_model="eval_f1_macro",
        greater_is_better=True,

        save_total_limit=2,
        fp16=True,
        report_to="none",
        label_names=["labels"],
        #gradient checkpointing
        gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False})


    time_callback = CheckpointTimeCallback(config["output_dir"])


    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=config["early_stopping_patience"]), time_callback])


#for resuming if something goes wrong
    last_checkpoint = None
    if os.path.isdir(config["output_dir"]):
        last_checkpoint = get_last_checkpoint(config["output_dir"])

    if last_checkpoint is not None:
        print(f"Resuming from checkpoint: {last_checkpoint}")
    else:
        print("Starting training from scratch.")


####imporved for the final run on MODERNBERT
    sync_cuda()
    # includes training + epoch validation + checkpoint saving + early stopping + loading best model
    trainer.train(resume_from_checkpoint=last_checkpoint)
    sync_cuda()
    # accumulated training time saved after completed checkpoints/epochs
    train_time_sec, current_session_train_time_sec = time_callback.get_times()


    if measure_vram:
        training_peak_vram_gb = get_peak_vram_gb()
    else:
        training_peak_vram_gb = None

    sync_cuda()
    val_start = time.perf_counter()

    val_results = trainer.evaluate(eval_dataset, metric_key_prefix="val")

    sync_cuda()
    val_eval_time_sec = time.perf_counter() - val_start

    result = {
        "model": config["base_model"],
        "dataset": "Scot-BESS",
        "method": "lora",
        "seed": seed,

        "learning_rate": config["learning_rate"],
        "micro_batch_size": config["micro_batch_size"],
        "gradient_accumulation_steps": config["gradient_accumulation_steps"],
        "effective_batch_size": config["effective_batch_size"],
        "num_train_epochs": config["num_train_epochs"],

        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "actual_epochs_trained": trainer.state.epoch,


        "train_time_sec": train_time_sec,
        "current_session_train_time_sec": current_session_train_time_sec,
        "val_eval_time_sec": val_eval_time_sec,

        "training_peak_vram_gb": training_peak_vram_gb,

        "trainable_params": trainable_params,
        "total_params": total_params,

        "val_f1_macro": val_results["val_f1_macro"],
        "val_f1_micro": val_results["val_f1_micro"]}

    if evaluate_test:
        sync_cuda()
        test_start = time.perf_counter()

        # single forward pass — gives metrics + raw predictions
        test_pred_output = trainer.predict(test_dataset)

        sync_cuda()
        test_eval_time_sec = time.perf_counter() - test_start

        # derive predictions - needed for classification report
        test_probs = 1 / (1 + np.exp(-test_pred_output.predictions))
        test_binary_preds = (test_probs >= 0.5).astype(int)
        gold_labels = (test_pred_output.label_ids >= 0.5).astype(int)

        test_metrics = test_pred_output.metrics

        result.update({
            "test_eval_time_sec": test_eval_time_sec,
            "test_inference_per_sample_ms": (test_eval_time_sec / len(test_dataset)) * 1000,
            "test_f1_macro": test_metrics["test_f1_macro"],
            "test_f1_micro": test_metrics["test_f1_micro"],
            "avg_predicted_labels": float(test_binary_preds.sum(axis=1).mean()),
            "avg_gold_labels": float(gold_labels.sum(axis=1).mean()),})

        if save_report:
            report_dict = classification_report(
                gold_labels, test_binary_preds,
                target_names=mlb.classes_, zero_division=0, output_dict=True)
            report_df = pd.DataFrame(report_dict).T
            report_path = os.path.join(output_dir, f"classification_report_seed_{seed}.csv")
            report_df.to_csv(report_path)
            print(f"Classification report saved to {report_path}")

            #saving raw arrays for possible future analysis
            predictions_path = os.path.join(
                output_dir,
                f"test_predictions_seed_{seed}.npz")
            np.savez_compressed(
                predictions_path,
                y_true=gold_labels,
                y_pred=test_binary_preds,
                y_prob=test_probs,
                label_names=np.array(mlb.classes_),
                threshold=np.array([0.5]))
            result["test_predictions_path"] = predictions_path
            print(f"Predictions saved to {predictions_path}")

    result["total_measured_time_sec"] = (result["train_time_sec"] + result["val_eval_time_sec"] + result.get("test_eval_time_sec", 0))

    return result

In [ ]:
#fixed params
base_config = {
    "output_dir": os.path.join(output_dir, "search"),
    "base_model": "answerdotai/ModernBERT-base",
    "tokenizer_name": "answerdotai/ModernBERT-base",

    "max_length": 8192,
    "num_train_epochs": 4, #fewer epochs for ModernBERT, for search only
    "weight_decay": 0.01,
    "warmup_ratio": 0.1,
    "early_stopping_patience": 3,
    "micro_batch_size": 4,

    #lora params
    "lora_r": 8,   #in Razuvayevskaya et al. (2024) they showed that this param value doesnt matter
    "lora_alpha": 8,          # alpha = r as in Razuvayevskaya et al. (2024)
    "lora_dropout": 0.05,     # as in Razuvayevskaya et al. (2024)
    "target_modules": ["Wqkv", "Wo", "Wi"], #Razuvayevskaya et al. (2024) - all layers
    "modules_to_save": ["head", "classifier"]}


learning_rates = [1e-4, 3e-4, 5e-4]
effective_batch_sizes = [8, 16]


search_results_path = os.path.join(output_dir, "search_results.csv")
search_results = []

for lr in learning_rates:
    for effective_bs in effective_batch_sizes:
        config = base_config.copy()
        config["learning_rate"] = lr
        config["effective_batch_size"] = effective_bs
        config["gradient_accumulation_steps"] = (effective_bs // config["micro_batch_size"])
        config["output_dir"] = (f"{base_config['output_dir']}/lr_{lr}_efbs_{effective_bs}")

        #skipping already-completed configs on resume
        if os.path.exists(search_results_path):
            existing = pd.read_csv(search_results_path)
            already_done = existing[
                (existing["learning_rate"] == lr) &
                (existing["effective_batch_size"] == effective_bs)]
            if len(already_done) > 0:
                print(f"Skipping lr={lr}, effective_bs={effective_bs} (already done)")
                search_results.append(already_done.iloc[0].to_dict())
                continue

        print("=" * 80)
        print(f"Running ModernBERT: lr={lr},  effective_batch_size={effective_bs}, grad_accum={config['gradient_accumulation_steps']}")
        print("=" * 80)

        result = run_training(config, seed=0)
        search_results.append(result)

        #saving incrementally after every config
        pd.DataFrame(search_results).to_csv(search_results_path, index=False)

search_results_df = pd.DataFrame(search_results)
search_results_df = search_results_df.sort_values("val_f1_macro", ascending=False).reset_index(drop=True)
search_results_df.to_csv(search_results_path, index=False)
#!!! train_time_sec in search results is unreliable due to checkpoint resumption !!!
# !!!timing is only reported from the final seed runs - I ensured the run is not resumed
search_results_df

Running ModernBERT: lr=0.0001,  effective_batch_size=8, grad_accum=2


model.safetensors: reconstructing file:   0%|          |  0.00B /  599MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


...

Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,2.015125,0.396286,0.685497,0.536361
2,1.272541,0.292075,0.783983,0.677089
3,0.957241,0.257144,0.808617,0.743159
4,0.790217,0.245778,0.826598,0.775584


[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
0.790217,0.245778,4,0.826598,0.775584


,model,dataset,method,seed,learning_rate,micro_batch_size,gradient_accumulation_steps,effective_batch_size,num_train_epochs,best_checkpoint,...,actual_epochs_trained,train_time_sec,current_session_train_time_sec,val_eval_time_sec,training_peak_vram_gb,trainable_params,total_params,val_f1_macro,val_f1_micro,total_measured_time_sec
0,answerdotai/ModernBERT-base,Scot-BESS,lora,0,0.0005,4,2,8,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,2051.752099,2051.752099,14.887017,None,2295572,151915816,0.813634,0.851344,2066.639116
1,answerdotai/ModernBERT-base,Scot-BESS,lora,0,0.0003,4,2,8,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,2051.356497,2051.356497,14.974508,None,2295572,151915816,0.784078,0.831795,2066.331006
2,answerdotai/ModernBERT-base,Scot-BESS,lora,0,0.0005,4,4,16,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,2051.421711,2051.421711,15.049061,None,2295572,151915816,0.775584,0.826598,2066.470772
3,answerdotai/ModernBERT-base,Scot-BESS,lora,0,0.0003,4,4,16,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,2045.589338,2045.589338,14.853288,None,2295572,151915816,0.713373,0.797116,2060.442625
4,answerdotai/ModernBERT-base,Scot-BESS,lora,0,0.0001,4,2,8,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,2058.683908,2058.683908,14.943511,None,2295572,151915816,0.660601,0.750000,2073.627419
5,answerdotai/ModernBERT-base,Scot-BESS,lora,0,0.0001,4,4,16,4,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,4.0,2048.904158,2048.904158,14.903957,None,2295572,151915816,0.574647,0.698193,2063.808116


In [ ]:
best_row = search_results_df.iloc[0]

best_lr = float(best_row["learning_rate"])
best_effective_batch_size = int(best_row["effective_batch_size"])
best_grad_accum = int(best_row["gradient_accumulation_steps"])

print("Best learning rate:", best_lr)
print("Best effective batch size:", best_effective_batch_size)
print("Gradient accumulation steps:", best_grad_accum)
print("Best validation macro-F1:", best_row["val_f1_macro"])
print("Best checkpoint:", best_row["best_checkpoint"])

Best learning rate: 0.0005
Best effective batch size: 8
Gradient accumulation steps: 2
Best validation macro-F1: 0.8136344543031957
Best checkpoint: /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_LoRA/search/lr_0.0005_efbs_8/checkpoint-672


In [ ]:
best_config = base_config.copy()
best_config["learning_rate"] = best_lr
best_config["effective_batch_size"] = best_effective_batch_size
best_config["gradient_accumulation_steps"] = best_grad_accum
best_config["selection_metric"] = "val_f1_macro"
best_config["best_validation_macro_f1"] = float(best_row["val_f1_macro"])
best_config["best_validation_micro_f1"] = float(best_row["val_f1_micro"])
best_config["best_checkpoint_from_search"] = best_row["best_checkpoint"]

best_config_path = os.path.join(output_dir, "best_config.json")

with open(best_config_path, "w") as f:
    json.dump(best_config, f, indent=2)


**Final run (test set) on the best found configuration**

In [ ]:
with open(best_config_path, "r") as f:
    final_config = json.load(f)

#final runs use the full training budget with early stopping
final_config["num_train_epochs"] = 10

In [ ]:
test_output_dir = os.path.join(output_dir, "test")
os.makedirs(test_output_dir, exist_ok=True)

test_results_path = os.path.join(test_output_dir, "SCOTBESS_ModernBERT_LoRA_test_results.csv")
test_results = []

for seed in [0, 1, 2]:
    config = final_config.copy()
    config["seed"] = seed
    config["output_dir"] = (os.path.join(test_output_dir, f"SCOTBESS_ModernBERT_LORA_test_seed_{seed}"))

    # Skip already-completed seeds on resume
    if os.path.exists(test_results_path):
        existing = pd.read_csv(test_results_path)
        already_done = existing[existing["seed"] == seed]
        if len(already_done) > 0:
            print(f"Skipping seed={seed} (already done)")
            test_results.append(already_done.iloc[0].to_dict())
            continue

    print("=" * 80)
    print(f"Final run: seed={seed}, lr={final_config['learning_rate']}, micro_batch={final_config['micro_batch_size']}, effective_batch={final_config['effective_batch_size']}, grad_accum={final_config['gradient_accumulation_steps']}")
    print("=" * 80)
    result = run_training(config, seed=seed, evaluate_test=True, measure_vram=True, save_report = True)
    test_results.append(result)

    # Save incrementally after every seed
    pd.DataFrame(test_results).to_csv(test_results_path, index=False)

test_results_df = pd.DataFrame(test_results)
test_results_df.to_csv(test_results_path, index=False)
test_results_df

Final run: seed=0, lr=0.0005, micro_batch=4, effective_batch=8, grad_accum=2


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Starting training from scratch.


Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,1.001046,0.383792,0.722195,0.624641
2,0.598894,0.264404,0.799370,0.694587
3,0.418639,0.250567,0.827068,0.781596
4,0.305792,0.220562,0.856725,0.824115
5,0.211310,0.227499,0.861167,0.831872
6,0.136779,0.244837,0.860887,0.835314
7,0.085191,0.241821,0.867005,0.848117
8,0.046661,0.247681,0.869609,0.857104
9,0.025013,0.250554,0.870045,0.853670
10,0.017279,0.252634,0.869609,0.852123


...

Epoch,Training Loss,Validation Loss,F1 Micro,F1 Macro
1,1.007170,0.380961,0.706488,0.576420
2,0.610524,0.266306,0.808155,0.706817
3,0.425240,0.250225,0.826131,0.778219
4,0.307035,0.233985,0.849929,0.826684
5,0.215052,0.232934,0.859274,0.840511
6,0.136052,0.228738,0.862646,0.839835
7,0.085326,0.244627,0.868787,0.846688
8,0.045845,0.244342,0.875313,0.855671
9,0.024954,0.249738,0.876518,0.856299
10,0.017405,0.247455,0.873727,0.853595


[transformers] early stopping required metric_for_best_model, but did not find eval_f1_macro so early stopping is disabled


Training Loss,Validation Loss,Epoch,F1 Micro,F1 Macro
0.017405,0.249738,10,0.876518,0.856299


Classification report saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_LoRA/classification_report_seed_2.csv
Predictions saved to /content/drive/MyDrive/thesis_results/SCOTBESS_EXPERIMENTS/SCOTBESS_ModernBERT_LoRA/test_predictions_seed_2.npz


,model,dataset,method,seed,learning_rate,micro_batch_size,gradient_accumulation_steps,effective_batch_size,num_train_epochs,best_checkpoint,...,val_f1_macro,val_f1_micro,test_eval_time_sec,test_inference_per_sample_ms,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,test_predictions_path,total_measured_time_sec
0,answerdotai/ModernBERT-base,Scot-BESS,lora,0,0.0005,4,2,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,0.857104,0.869609,15.453509,90.902995,0.837415,0.865298,5.829412,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,5173.900175
1,answerdotai/ModernBERT-base,Scot-BESS,lora,1,0.0005,4,2,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,0.844639,0.865835,15.398596,90.579979,0.830450,0.861292,5.829412,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,5169.581251
2,answerdotai/ModernBERT-base,Scot-BESS,lora,2,0.0005,4,2,8,10,/content/drive/MyDrive/thesis_results/SCOTBESS...,...,0.856299,0.876518,15.521595,91.303500,0.847273,0.870091,5.764706,5.917647,/content/drive/MyDrive/thesis_results/SCOTBESS...,5168.244565


In [ ]:
test_summary_df = test_results_df[[
    "test_f1_macro",
    "test_f1_micro",
    "avg_predicted_labels",
    "avg_gold_labels",
    "train_time_sec",
    "val_eval_time_sec",
    "test_eval_time_sec",
    "test_inference_per_sample_ms",
    "training_peak_vram_gb",
    "actual_epochs_trained",
    "trainable_params",
    "total_params",
    "total_measured_time_sec"]].agg(["mean", "std"])

test_summary_path =  os.path.join(test_output_dir, "SCOTBESS_ModernBERT_LoRA_test_results_summary.csv")
test_summary_df.to_csv(test_summary_path)

test_summary_df

,test_f1_macro,test_f1_micro,avg_predicted_labels,avg_gold_labels,train_time_sec,val_eval_time_sec,test_eval_time_sec,test_inference_per_sample_ms,training_peak_vram_gb,actual_epochs_trained,trainable_params,total_params,total_measured_time_sec
mean,0.838379,0.865560,5.807843,5.917647,5140.178672,14.938758,15.457900,90.928825,5.437192,10.0,2295572.0,151915816.0,5170.575330
std,0.008453,0.004405,0.037358,0.000000,2.987358,0.090515,0.061617,0.362452,0.000269,0.0,0.0,0.0,2.955948


In [ ]:
from google.colab import runtime
runtime.unassign()